# CareLoop Clinics — Part 4: GenAI follow-up analysis

Use Gemini Flash with JSON-only structured output and validate the response using Pydantic. Set `GEMINI_API_KEY` in the runtime environment; never paste the key into a notebook cell or commit it. The first 10 notes are processed. Blank notes are skipped without an API call.\n\nThe checked-in `outputs/notes_analysed.csv` is explicitly marked `offline_rule_based`; it is not a Gemini result. Run the API cell below with a valid key to replace it with actual model responses.

In [ ]:
from pathlib import Path
import sys, os, pandas as pd
sys.path.insert(0,str(Path.cwd()))
from src.pipeline import OUT, FollowupInsight, ALLOWED, run_part4, merge_recommendation

## Task 4.2 — Process ten notes
The pipeline retains each raw response, including invalid responses. Invalid replies are not repaired or dropped.

In [ ]:
analysed = run_part4()
analysed.head(10)

## Task 4.3 — Deliberately bad replies
The two replies below test confidence upper-bound validation and a missing required field.

In [ ]:
from pydantic import ValidationError
bad_replies = [
    {'root_cause':'other','confidence':1.7,'summary':'test reply','recommended_action':'call'},
    {'root_cause':'other','confidence':0.5,'summary':'missing action'}
]
for reply in bad_replies:
    try: FollowupInsight.model_validate(reply)
    except ValidationError as e: print(type(e).__name__, str(e).splitlines()[0])

## Task 4.4 — Merge and recommendation dataset

In [ ]:
merged = merge_recommendation()
print('Merged rows:', len(merged))
merged.head()

For the prompt comparison, call `analyze_note(note, client, prompt_version='v1')` and `prompt_version='v2'` on the same two example notes, then compare structured root-cause outputs. Keep API calls within the free-tier limits.

## Task 4.3(b) — Compare V1 and V2 prompts on N007 and N009

In [ ]:
from google import genai
import os, json
assert os.environ.get('GEMINI_API_KEY'), 'Set GEMINI_API_KEY before running this cell.'
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
sample = pd.read_csv(Path('data/followup_notes.csv')).set_index('note_id')
comparison = []
for note_id in ['N007', 'N009']:
    note_text = str(sample.loc[note_id, 'note_text'])
    for version in ['v1', 'v2']:
        raw = analyze_note(note_text, client, prompt_version=version)
        try:
            parsed = FollowupInsight.model_validate(json.loads(raw))
            if parsed.root_cause not in ALLOWED: raise ValueError('root_cause not allowed')
            comparison.append({'note_id': note_id, 'prompt': version, 'root_cause': parsed.root_cause, 'valid': True, 'raw_response': raw})
        except Exception as exc:
            comparison.append({'note_id': note_id, 'prompt': version, 'root_cause': '', 'valid': False, 'raw_response': raw, 'error': str(exc)})
pd.DataFrame(comparison)
